# 🧩 Mini-Lab: Evaluation Self-Check

**Module: Is It Any Good? Quality, Latency, Cost** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Write** an evidence-based ship / don't-ship decision that states its accepted trade-offs
2. **Identify** what evidence a complete evaluation report must contain before a decision is defensible
3. **Recognize** when an evaluation has gaps that would make a ship decision premature

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Evidence-Based Decision | "I would ship V2 because ___, accepting ___" is an evidence-based launch call that ties measured quality, latency, and cost to explicit risk tradeoffs. |
| Gold Set | A small, verified labeled sample used as the trusted reference for evaluation. It gives high-signal quality checks, but small sample sizes can mislead, so treat results as estimates, not absolute truth. |
| Percentile Latency | The mean hides the slow calls and timeouts, so use percentiles like p50 and p95 to describe typical latency and tail slowness users actually feel. |
| Cost-Per-Unit Estimate | Compute cost per 1,000 records from your own logged counts, then treat price as a variable so you can compare models and prompts fairly. |
| Version Control Basics | Init a repo, ignore generated files, commit small changes, and tag v0 to v5 as known good checkpoints you can reset to safely. |


## What Makes a Ship Decision Defensible?

In `lab-prompt-ab-evaluation`, you built a full evaluation pipeline and wrote a ship decision at the end. This mini-lab zooms in on **the decision itself** — what must it contain, and how do you check that nothing is missing?

A ship decision is not "it looks good, let's deploy." It's a **self-check** — a structured document where every claim is backed by a number, every trade-off is named, and every limitation is stated honestly.

Think of it as a checklist you run *on your own evaluation* before presenting it to anyone.

## Step 1 — The Evaluation Checklist

Before writing a ship decision, verify that you have all the required evidence. Here's the checklist — every item must have a concrete value, not just "we checked this."

In [1]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

checklist = """
### ✅ Evaluation Self-Check Checklist

| # | Evidence Required | Why It Matters | Your Value |
|---|---|---|---|
| 1 | **Gold set size** | Determines how much you can trust your metrics | ___ rows |
| 2 | **Baseline accuracy** | Proves the AI adds value over a dumb predictor | ___% |
| 3 | **System accuracy** | Overall correctness on the gold set | ___% |
| 4 | **Per-class precision & recall** | Reveals where the system fails (accuracy hides this) | per class |
| 5 | **p50 latency** | Typical user experience | ___ ms |
| 6 | **p95 latency** | Worst reasonable user experience | ___ ms |
| 7 | **Cost per 1,000 records** | Is this viable at production scale? | $___ |
| 8 | **Prompt / model version** | Which exact configuration produced these results? | v___ |
| 9 | **Trade-offs accepted** | What gets worse and why that's OK | stated |
| 10 | **Sample-size caveat** | Honest statement of confidence limits | stated |

> **Rule:** If any row is blank, the ship decision is not ready.
"""
md(checklist)


### ✅ Evaluation Self-Check Checklist

| # | Evidence Required | Why It Matters | Your Value |
|---|---|---|---|
| 1 | **Gold set size** | Determines how much you can trust your metrics | ___ rows |
| 2 | **Baseline accuracy** | Proves the AI adds value over a dumb predictor | ___% |
| 3 | **System accuracy** | Overall correctness on the gold set | ___% |
| 4 | **Per-class precision & recall** | Reveals where the system fails (accuracy hides this) | per class |
| 5 | **p50 latency** | Typical user experience | ___ ms |
| 6 | **p95 latency** | Worst reasonable user experience | ___ ms |
| 7 | **Cost per 1,000 records** | Is this viable at production scale? | $___ |
| 8 | **Prompt / model version** | Which exact configuration produced these results? | v___ |
| 9 | **Trade-offs accepted** | What gets worse and why that's OK | stated |
| 10 | **Sample-size caveat** | Honest statement of confidence limits | stated |

> **Rule:** If any row is blank, the ship decision is not ready.


## Step 2 — A Filled-In Example

Let's fill in the checklist with concrete numbers from an evaluation run. We'll use results similar to what you saw in `lab-prompt-ab-evaluation` — a support-ticket classifier being evaluated for a prompt upgrade.

In [2]:
# Evaluation results from a completed run
eval_results = {
    "gold_set_size": 40,
    "baseline_accuracy": 45.0,      # majority-class baseline
    "system_accuracy": 80.0,         # prompt V2
    "per_class": {
        "billing":   {"precision": 0.82, "recall": 0.78},
        "technical": {"precision": 0.79, "recall": 0.85},
        "account":   {"precision": 0.78, "recall": 0.78},
    },
    "p50_latency_ms": 330,
    "p95_latency_ms": 1850,
    "cost_per_1000": 0.0554,
    "prompt_version": "V2",
    "model": "gpt-4o-mini",
}

# Fill in the checklist
r = eval_results
pc = r["per_class"]

precision_recall_rows = ""
for cls in ["billing", "technical", "account"]:
    precision_recall_rows += (
        f"| | {cls} | P={pc[cls]['precision']:.0%}  R={pc[cls]['recall']:.0%} |\n"
    )

filled = f"""
### ✅ Evaluation Self-Check — Filled In

| # | Evidence Required | Value |
|---|---|---|
| 1 | **Gold set size** | {r['gold_set_size']} rows |
| 2 | **Baseline accuracy** | {r['baseline_accuracy']:.1f}% (majority-class) |
| 3 | **System accuracy** | {r['system_accuracy']:.1f}% |
| 4 | **Margin over baseline** | +{r['system_accuracy'] - r['baseline_accuracy']:.1f} pp |
{precision_recall_rows}\
| 5 | **p50 latency** | {r['p50_latency_ms']} ms |
| 6 | **p95 latency** | {r['p95_latency_ms']} ms |
| 7 | **Cost per 1,000** | ${r['cost_per_1000']:.4f} |
| 8 | **Version** | Prompt {r['prompt_version']}, model `{r['model']}` |

All rows filled → we can proceed to the ship decision.
"""
md(filled)


### ✅ Evaluation Self-Check — Filled In

| # | Evidence Required | Value |
|---|---|---|
| 1 | **Gold set size** | 40 rows |
| 2 | **Baseline accuracy** | 45.0% (majority-class) |
| 3 | **System accuracy** | 80.0% |
| 4 | **Margin over baseline** | +35.0 pp |
| | billing | P=82%  R=78% |
| | technical | P=79%  R=85% |
| | account | P=78%  R=78% |
| 5 | **p50 latency** | 330 ms |
| 6 | **p95 latency** | 1850 ms |
| 7 | **Cost per 1,000** | $0.0554 |
| 8 | **Version** | Prompt V2, model `gpt-4o-mini` |

All rows filled → we can proceed to the ship decision.


## Step 3 — Write the Ship Decision

A defensible ship decision has **five parts**:

1. **Question** — What are you deciding?
2. **Evidence** — The metrics (not opinions)
3. **What gets better / worse** — Trade-offs named explicitly
4. **Recommendation** — Ship, don't ship, or conditional
5. **Caveats** — Sample size, conditions to revisit

Let's generate one from the results above.

In [3]:
r = eval_results
pc = r["per_class"]

decision = f"""### 📋 Ship Decision: Deploy Prompt {r['prompt_version']} with `{r['model']}`

**Question:** Is the support-ticket classifier ready to deploy with Prompt {r['prompt_version']}?

---

#### Evidence

| Metric | Value |
|--------|-------|
| Gold set | {r['gold_set_size']} rows |
| Baseline accuracy | {r['baseline_accuracy']:.1f}% |
| System accuracy | {r['system_accuracy']:.1f}% (+{r['system_accuracy'] - r['baseline_accuracy']:.1f} pp over baseline) |
| billing P/R | {pc['billing']['precision']:.0%} / {pc['billing']['recall']:.0%} |
| technical P/R | {pc['technical']['precision']:.0%} / {pc['technical']['recall']:.0%} |
| account P/R | {pc['account']['precision']:.0%} / {pc['account']['recall']:.0%} |
| p50 latency | {r['p50_latency_ms']} ms |
| p95 latency | {r['p95_latency_ms']} ms |
| Cost per 1,000 | ${r['cost_per_1000']:.4f} |
| Configuration | Prompt {r['prompt_version']}, `{r['model']}` |

#### What Gets Better

- System accuracy is **{r['system_accuracy'] - r['baseline_accuracy']:.1f} pp above baseline** — the classifier adds clear value over random guessing.
- All three classes have recall ≥ 78%, meaning no category is being systematically missed.
- Cost is low at ${r['cost_per_1000']:.4f} per 1,000 records (${r['cost_per_1000'] * 100:.2f} per 100K).

#### What Gets Worse / Accepted Trade-offs

- **p95 latency is {r['p95_latency_ms']} ms** — about 1 in 20 requests takes nearly 2 seconds. This is acceptable for a background classification job but would need investigation for a user-facing feature.
- **Billing precision at {pc['billing']['precision']:.0%}** means ~18% of tickets predicted as billing are actually something else. Acceptable for routing (human reviews follow), but not for automated billing actions.

#### Recommendation

**Ship** for background ticket routing. The system beats the baseline by a wide margin across all classes. Latency and cost are within acceptable bounds for a non-real-time use case.

#### Caveats

- ⚠️ Gold set is only **{r['gold_set_size']} rows** — metrics are directional estimates. Expand to 80–100 rows before treating precision/recall differences as reliable.
- Monitor billing precision in the first week of production. If it drops below 75%, revisit the prompt.
- These results are tied to **Prompt {r['prompt_version']}** on **`{r['model']}`**. Changing either requires re-evaluation.
"""

md(decision)

### 📋 Ship Decision: Deploy Prompt V2 with `gpt-4o-mini`

**Question:** Is the support-ticket classifier ready to deploy with Prompt V2?

---

#### Evidence

| Metric | Value |
|--------|-------|
| Gold set | 40 rows |
| Baseline accuracy | 45.0% |
| System accuracy | 80.0% (+35.0 pp over baseline) |
| billing P/R | 82% / 78% |
| technical P/R | 79% / 85% |
| account P/R | 78% / 78% |
| p50 latency | 330 ms |
| p95 latency | 1850 ms |
| Cost per 1,000 | $0.0554 |
| Configuration | Prompt V2, `gpt-4o-mini` |

#### What Gets Better

- System accuracy is **35.0 pp above baseline** — the classifier adds clear value over random guessing.
- All three classes have recall ≥ 78%, meaning no category is being systematically missed.
- Cost is low at $0.0554 per 1,000 records ($5.54 per 100K).

#### What Gets Worse / Accepted Trade-offs

- **p95 latency is 1850 ms** — about 1 in 20 requests takes nearly 2 seconds. This is acceptable for a background classification job but would need investigation for a user-facing feature.
- **Billing precision at 82%** means ~18% of tickets predicted as billing are actually something else. Acceptable for routing (human reviews follow), but not for automated billing actions.

#### Recommendation

**Ship** for background ticket routing. The system beats the baseline by a wide margin across all classes. Latency and cost are within acceptable bounds for a non-real-time use case.

#### Caveats

- ⚠️ Gold set is only **40 rows** — metrics are directional estimates. Expand to 80–100 rows before treating precision/recall differences as reliable.
- Monitor billing precision in the first week of production. If it drops below 75%, revisit the prompt.
- These results are tied to **Prompt V2** on **`gpt-4o-mini`**. Changing either requires re-evaluation.


## Step 4 — Spot the Missing Evidence

Now let's practice the critical skill: **catching when a ship decision is incomplete.** Below is a decision report with deliberate gaps. Can the checklist catch them?

In [4]:
# An incomplete evaluation report — on purpose
incomplete_report = {
    "gold_set_size": None,           # never stated!
    "baseline_accuracy": None,       # never computed!
    "system_accuracy": 82.0,
    "per_class": None,               # only overall accuracy reported
    "p50_latency_ms": 280,
    "p95_latency_ms": None,          # only p50 reported
    "cost_per_1000": 0.0480,
    "prompt_version": None,          # "the latest prompt" — which one?
    "model": "gpt-4o-mini",
}

# Run the self-check
checks = [
    ("Gold set size",           incomplete_report["gold_set_size"]),
    ("Baseline accuracy",       incomplete_report["baseline_accuracy"]),
    ("System accuracy",         incomplete_report["system_accuracy"]),
    ("Per-class precision/recall", incomplete_report["per_class"]),
    ("p50 latency",             incomplete_report["p50_latency_ms"]),
    ("p95 latency",             incomplete_report["p95_latency_ms"]),
    ("Cost per 1,000",          incomplete_report["cost_per_1000"]),
    ("Prompt version",          incomplete_report["prompt_version"]),
]

print("Self-Check Results:\n")
missing = 0
for name, value in checks:
    if value is None:
        print(f"  ❌ {name}: MISSING")
        missing += 1
    else:
        print(f"  ✅ {name}: {value}")

print(f"\n{'='*40}")
if missing > 0:
    print(f"🚫 {missing} item(s) missing — this evaluation is NOT ready for a ship decision.")
else:
    print("✅ All items present — ready to write the ship decision.")

Self-Check Results:

  ❌ Gold set size: MISSING
  ❌ Baseline accuracy: MISSING
  ✅ System accuracy: 82.0
  ❌ Per-class precision/recall: MISSING
  ✅ p50 latency: 280
  ❌ p95 latency: MISSING
  ✅ Cost per 1,000: 0.048
  ❌ Prompt version: MISSING

🚫 5 item(s) missing — this evaluation is NOT ready for a ship decision.


**Why each gap matters:**

- **No gold set size** → We can't assess confidence. 82% on 10 rows means very little; 82% on 200 rows is solid.
- **No baseline** → 82% accuracy sounds good, but if the majority class is 80%, the AI barely beats random guessing.
- **No per-class metrics** → Overall 82% could mean 95% on billing and 40% on account. You'd never know.
- **No p95 latency** → p50 of 280 ms looks fast, but if p95 is 5,000 ms, 1 in 20 users waits 5 seconds.
- **No prompt version** → If something breaks, you can't reproduce the evaluation or roll back. This is **version control** — knowing exactly which prompt + model produced which results.

## Step 5 — Version Control: Tag Your Evaluation

Every ship decision should record **exactly** which configuration produced the results. This isn't about Git commits (though those help) — it's about the minimum version metadata that makes an evaluation reproducible.

In [5]:
from datetime import datetime

eval_tag = {
    "eval_id": "eval-2024-001",
    "date": datetime.now().strftime("%Y-%m-%d"),
    "prompt_version": "V2",
    "model": "gpt-4o-mini",
    "gold_set_version": "gold-v1 (40 rows)",
    "evaluator": "your-name",
    "decision": "SHIP for background routing",
}

tag_rows = "\n".join(f"| **{k}** | {v} |" for k, v in eval_tag.items())

md(f"""### 🏷️ Evaluation Tag

Save this alongside your ship decision so anyone can trace the results back to the exact setup.

| Field | Value |
|-------|-------|
{tag_rows}

> **Tip:** Store this tag in a file (e.g., `eval-2024-001.json`) or in your project's version control.
> If the prompt or model changes, a new evaluation with a new tag is required.
""")

### 🏷️ Evaluation Tag

Save this alongside your ship decision so anyone can trace the results back to the exact setup.

| Field | Value |
|-------|-------|
| **eval_id** | eval-2024-001 |
| **date** | 2026-09-16 |
| **prompt_version** | V2 |
| **model** | gpt-4o-mini |
| **gold_set_version** | gold-v1 (40 rows) |
| **evaluator** | your-name |
| **decision** | SHIP for background routing |

> **Tip:** Store this tag in a file (e.g., `eval-2024-001.json`) or in your project's version control.
> If the prompt or model changes, a new evaluation with a new tag is required.


## 🎯 Summary

### Key Takeaways

1. **Evidence-based decision** — a ship decision must state the question, present metrics, name trade-offs, make a recommendation, and flag caveats; "it looks good" is not a decision
2. **Self-check checklist** — before writing any recommendation, verify that gold set size, baseline, per-class metrics, p50/p95 latency, cost, and version info are all present; any gap makes the decision premature
3. **Sample-size honesty** — always state the gold set size and acknowledge that small sets produce directional estimates, not definitive proof
4. **Version tagging** — record which prompt version, model, and gold set produced the results so the evaluation is reproducible and you can roll back if needed